# mine-tuning-model (RTX 5060 Ti 최적화 버전)

## 사전 준비 (터미널에서 먼저 실행)

```bash
# 1. 가상환경 생성 및 활성화
python -m venv venv
source venv/Scripts/activate

# 2. PyTorch 설치 — RTX 5060 Ti (Blackwell)는 CUDA 12.8 필요!
#    nvidia-smi 실행해서 CUDA 버전 확인 후 아래 명령어 선택

# ✅ CUDA 12.8 (RTX 5060 Ti 권장)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

# 3. Jupyter 설치
pip install jupyter

# vs code 커널 등록
python -m ipykernel install --user --name venv --display-name "mine-tuning (venv)"

# 4. 노트북 실행
jupyter notebook
```

## 0. GPU 확인 (제일 먼저 실행!)

In [1]:
import subprocess
import sys

# GPU 확인
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except FileNotFoundError:
    print('⚠️  nvidia-smi를 찾을 수 없습니다. NVIDIA 드라이버가 설치되어 있는지 확인하세요.')

# PyTorch CUDA 확인
import torch
print(f'PyTorch 버전: {torch.__version__}')
print(f'CUDA 버전:    {torch.version.cuda}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU 이름: {gpu_name}')
    print(f'GPU VRAM: {vram_gb:.1f}GB')

    # RTX 5060 Ti는 bf16 지원 확인
    bf16_support = torch.cuda.is_bf16_supported()
    print(f'bf16 지원: {bf16_support}')  # True여야 정상
else:
    print('❌ CUDA를 사용할 수 없습니다.')
    print('   → 사전 준비 단계에서 CUDA 12.8 버전 PyTorch를 설치했는지 확인하세요.')
    sys.exit('GPU가 필요합니다.')

No devices were found

PyTorch 버전: 2.11.0+cu128
CUDA 버전:    12.8
CUDA 사용 가능: False
❌ CUDA를 사용할 수 없습니다.
   → 사전 준비 단계에서 CUDA 12.8 버전 PyTorch를 설치했는지 확인하세요.


c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\torch\cuda\__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 1: invalid argument (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10\cuda\CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


SystemExit: GPU가 필요합니다.

c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 1. 필수 라이브러리 설치

> 💡 처음 한 번만 실행하면 돼요!

In [ ]:
import subprocess
subprocess.run([
    'pip', 'install',
    'transformers==4.51.3',
    'trl==0.17.0',
    'datasets',
    'peft',
    'accelerate',
    'bitsandbytes',  # 최신 버전은 Windows 공식 지원
    '-q'
], check=True)

print('✅ 패키지 설치 완료')

## 2. 데이터 로드 및 Instruction 형식 변환

In [ ]:
from datasets import load_dataset

# 데이터 로드
ds = load_dataset('ddorin/minecraft-question-answer-260k')
print(ds)

# Instruction 형식으로 변환
def format_instruction(example):
    return {
        'text': f"""<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
{example['question']}
<|im_end|>
<|im_start|>assistant
{example['answer']}
<|im_end|>"""
    }

train_data = ds['train'].map(format_instruction, remove_columns=['question', 'answer', 'source'])
print(f'✅ 데이터 변환 완료: {len(train_data)}개')
print('\n샘플 확인:')
print(train_data[0]['text'])

## 3. 모델, 토크나이저 로드

> 💡 4bit 양자화로 로드해서 VRAM을 절약해요. 7B 모델이 약 5~6GB로 줄어들어요.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc

# VRAM 비우기
gc.collect()
torch.cuda.empty_cache()

model_id = 'Qwen/Qwen2.5-7B-Instruct'

# 4bit 양자화 설정
# RTX 5060 Ti (Blackwell)는 bf16 지원 → compute_dtype을 bf16으로 설정!
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,   # ✅ 5060 Ti는 bf16 사용 (fp16보다 안정적)
    bnb_4bit_use_double_quant=True,
)

print(f'모델 로딩 중: {model_id}')
print('처음 실행 시 약 15GB 다운로드가 필요해요...')

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={'': 0},  # GPU 0번에 전체 로드
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

used  = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\n✅ 모델 로드 완료!')
print(f'VRAM 사용: {used:.1f}GB / {total:.1f}GB')

## 4. LoRA 어댑터 설정

> 💡 전체 모델을 학습하는 게 아니라 LoRA라는 작은 어댑터만 학습해요.
> 덕분에 VRAM을 훨씬 적게 쓰면서도 파인튜닝 효과를 얻을 수 있어요!

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4bit 학습 준비
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj',
        'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 전체 파라미터 중 약 1~2%만 학습된다고 나오면 정상이에요!

## 5. 학습 실행 (SFTTrainer)

### ⚙️ 매 세션마다 아래 변수만 수정하세요

| 회차 | CHUNK_NUM | CHUNK_START | CHUNK_END |
|------|-----------|-------------|----------|
| 1회차 | 1 | 0 | 130000 |
| 2회차 | 2 | 130000 | 268239 |

In [ ]:
from trl import SFTTrainer, SFTConfig
from peft import PeftModel
import os

# ============================================================
# 매 세션마다 이 부분만 수정!
# ============================================================
CHUNK_NUM   = 1           # 1회차: 1  /  2회차: 2
CHUNK_START = 0           # 1회차: 0  /  2회차: 130000
CHUNK_END   = 130000      # 1회차: 130000  /  2회차: 268239

OUTPUT_DIR  = r'C:\mine-tuning-output'  # 저장 경로 (한글/공백 없는 경로 권장)
# OUTPUT_DIR = './mine-tuning-output'   # 현재 폴더에 저장하려면 이걸 사용
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)
chunk_dir = os.path.join(OUTPUT_DIR, f'chunk{CHUNK_NUM}')

# 2회차 이상: 이전 청크 모델 불러오기
if CHUNK_NUM > 1:
    prev_dir = os.path.join(OUTPUT_DIR, f'chunk{CHUNK_NUM - 1}')
    print(f'이전 모델 로드 중: {prev_dir}')
    model = PeftModel.from_pretrained(model, prev_dir)
    model = model.merge_and_unload()
    model = get_peft_model(model, lora_config)
    print('✅ 이전 모델 병합 완료!')

# 청크 데이터 선택
train_chunk = train_data.select(range(CHUNK_START, CHUNK_END))
print(f'📦 {CHUNK_NUM}회차 학습 데이터: {len(train_chunk):,}건 ({CHUNK_START}~{CHUNK_END})')

# 학습 설정
# ✅ RTX 5060 Ti 최적화:
#    - bf16=True  (Blackwell 아키텍처 지원, fp16보다 안정적)
#    - fp16=False (bf16 쓸 때 fp16은 꺼야 해요)
#    - per_device_train_batch_size=4 (16GB VRAM 기준 적정값)
training_args = SFTConfig(
    output_dir=chunk_dir,
    num_train_epochs=1,
    per_device_train_batch_size=4,   # VRAM 부족 시 2로 줄이고 gradient_accumulation_steps=8로 올리기
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,                       # ✅ RTX 5060 Ti는 bf16 사용
    fp16=False,                      # ✅ bf16 쓸 때 fp16은 반드시 False
    logging_steps=50,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    warmup_steps=100,
    lr_scheduler_type='cosine',
    report_to='none',
    dataloader_num_workers=0,        # ✅ Windows에서는 반드시 0 (멀티프로세싱 오류 방지)
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_chunk,
    args=training_args,
    processing_class=tokenizer,
)

print(f'🚀 {CHUNK_NUM}회차 학습 시작!')
trainer.train()

# 저장
trainer.model.save_pretrained(chunk_dir)
tokenizer.save_pretrained(chunk_dir)
print(f'\n✅ {CHUNK_NUM}회차 저장 완료! → {chunk_dir}')

## 6. 버전 확인

In [ ]:
import transformers, trl, torch

print(f'transformers: {transformers.__version__}')
print(f'trl:          {trl.__version__}')
print(f'torch:        {torch.__version__}')
print(f'CUDA:         {torch.version.cuda}')

---

## 📋 RTX 5060 Ti 실행 시 주의사항

| 항목 | 내용 |
|------|------|
| **PyTorch** | CUDA 12.8 버전으로 설치 (`--index-url .../cu128`) |
| **bf16** | RTX 5060 Ti (Blackwell)는 `bf16=True` 사용 가능 ✅ |
| **fp16** | bf16 사용 시 반드시 `fp16=False` |
| **bitsandbytes** | `pip install bitsandbytes` (최신 버전 Windows 공식 지원) |
| **dataloader_num_workers** | 반드시 `0`으로 설정 (Windows 멀티프로세싱 이슈) |
| **경로** | `OUTPUT_DIR`에 한글/공백이 없는 경로 사용 권장 |
| **VRAM 부족 시** | `per_device_train_batch_size=2`, `gradient_accumulation_steps=8` |

### 🐛 자주 나오는 에러

```
CUDA error: no kernel image is available for execution on the device
```
→ PyTorch가 5060 Ti를 지원하지 않는 버전이에요. cu128 버전으로 재설치하세요.

```
bitsandbytes: CUDA Setup failed
```
→ `pip install bitsandbytes --upgrade` 로 최신 버전으로 업그레이드하세요.